# 🏎️ F1 Season Monte Carlo Simulator — Results Visualization

This notebook loads the output CSVs from the Monte Carlo simulation and visualizes the probability distributions for each driver across all championship finishing positions.

Each CSV represents 10,000+ simulated seasons. Values in each cell are the **probability** (0–1) that a given driver finishes the championship in that position.

---

## 0. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

# Paths — relative to notebook location (notebooks/ folder)
OUTPUT_DIR = Path("..") / "csv_files"

# Plot styling
plt.rcParams['figure.facecolor'] = '#0f0f0f'
plt.rcParams['axes.facecolor'] = '#1a1a1a'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'
plt.rcParams['grid.color'] = '#2a2a2a'
plt.rcParams['font.family'] = 'monospace'
plt.rcParams['figure.dpi'] = 120

F1_RED = '#E8002D'
F1_SILVER = '#C0C0C0'

print('Setup complete.')

ModuleNotFoundError: No module named 'pandas'

## 1. Load Data

In [ ]:
def load_sim_csv(filename):
    """Load a simulation output CSV. Index = driver codes, columns = finishing positions + DNF."""
    df = pd.read_csv(OUTPUT_DIR / filename, index_col=0)
    # Ensure column names are strings for consistent handling
    df.columns = df.columns.astype(str)
    return df

# Load both outputs — update filenames if yours differ
standings_prob = load_sim_csv("2026_Monaco Grand Prix_predictions.csv")
simulated_standings = load_sim_csv("2026_post_Canadian Grand Prix_prediction.csv")

print(f"Loaded standings_prob:       {standings_prob.shape[0]} drivers x {standings_prob.shape[1]} positions")
print(f"Loaded simulated_standings:  {simulated_standings.shape[0]} drivers x {simulated_standings.shape[1]} positions")
print(f"\nDrivers: {list(standings_prob.index)}")

## 2. Quick Sanity Check

In [ ]:
# Each driver's probabilities should sum to ~1.0
row_sums = standings_prob.sum(axis=1)
print("Row sums (should be ~1.0 for each driver):")
print(row_sums.round(4).to_string())

# Each position column should also sum to ~1.0 (excluding DNF)
numeric_cols = [c for c in standings_prob.columns if c != 'DNF']
col_sums = standings_prob[numeric_cols].sum(axis=0)
print(f"\nColumn sums (positions 1–{len(numeric_cols)}, should each be ~1.0):")
print(col_sums.round(4).to_string())

## 3. Championship Win Probability — Bar Chart

P(driver finishes P1 in the championship) for all drivers, sorted descending.

In [ ]:
win_probs = standings_prob['1'].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))

colors = [F1_RED if i == 0 else F1_SILVER for i in range(len(win_probs))]
bars = ax.bar(win_probs.index, win_probs.values * 100, color=colors, edgecolor='none', width=0.65)

# Value labels on bars
for bar, val in zip(bars, win_probs.values):
    if val >= 0.01:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.4,
            f'{val*100:.1f}%',
            ha='center', va='bottom', fontsize=8, color='white'
        )

ax.set_title('Championship Win Probability — Monte Carlo (10,000 simulations)', 
             fontsize=13, pad=15, color='white')
ax.set_ylabel('Probability (%)', fontsize=10)
ax.set_xlabel('Driver', fontsize=10)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.grid(axis='y', linewidth=0.5)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'win_probability.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('Saved: output/win_probability.png')

## 4. Full Probability Heatmap — All Drivers, All Positions

Drivers sorted by expected finishing position (probability-weighted mean). Higher probability = darker red.

In [ ]:
# Sort drivers by probability-weighted expected position
numeric_cols = [c for c in standings_prob.columns if c.isdigit()]
positions = np.array([int(c) for c in numeric_cols])

expected_pos = standings_prob[numeric_cols].apply(
    lambda row: np.dot(row.values, positions), axis=1
)
sorted_drivers = expected_pos.sort_values().index
heatmap_data = standings_prob.loc[sorted_drivers, numeric_cols]

fig, ax = plt.subplots(figsize=(18, 10))

sns.heatmap(
    heatmap_data,
    ax=ax,
    cmap=sns.color_palette('rocket_r', as_cmap=True),
    linewidths=0.3,
    linecolor='#0f0f0f',
    cbar_kws={'label': 'Probability', 'shrink': 0.6},
    fmt='.0%',
    annot=heatmap_data.applymap(lambda x: f'{x*100:.0f}%' if x >= 0.04 else ''),
    annot_kws={'size': 7, 'color': 'white'},
    vmin=0,
    vmax=heatmap_data.values.max()
)

ax.set_title('Championship Finishing Position Probability — All Drivers\n(sorted by expected position)',
             fontsize=13, pad=15, color='white')
ax.set_xlabel('Championship Position', fontsize=10)
ax.set_ylabel('Driver', fontsize=10)
ax.tick_params(axis='both', labelsize=9)

# Colorbar label color
ax.collections[0].colorbar.ax.yaxis.label.set_color('white')
ax.collections[0].colorbar.ax.tick_params(colors='white')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'full_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('Saved: output/full_heatmap.png')

## 5. Podium Probability — Top 3 Finish

P(driver finishes P1, P2, or P3 in the championship), sorted by total podium probability.

In [ ]:
podium_prob = (standings_prob['1'] + standings_prob['2'] + standings_prob['3']).sort_values(ascending=True)

# Stacked bar: P1 / P2 / P3 contributions
p1 = standings_prob.loc[podium_prob.index, '1']
p2 = standings_prob.loc[podium_prob.index, '2']
p3 = standings_prob.loc[podium_prob.index, '3']

fig, ax = plt.subplots(figsize=(10, 9))

ax.barh(podium_prob.index, p1 * 100, color=F1_RED,    label='P1 — Champion',  height=0.6)
ax.barh(podium_prob.index, p2 * 100, left=p1 * 100,   color='#C0C0C0', label='P2',           height=0.6)
ax.barh(podium_prob.index, p3 * 100, left=(p1+p2)*100, color='#CD7F32', label='P3',           height=0.6)

# Total label at end of each bar
for driver in podium_prob.index:
    total = podium_prob[driver] * 100
    ax.text(total + 0.5, driver, f'{total:.1f}%', va='center', fontsize=8, color='white')

ax.set_title('Podium Finish Probability (P1 + P2 + P3)\nStacked by position contribution',
             fontsize=13, pad=15, color='white')
ax.set_xlabel('Probability (%)', fontsize=10)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend(loc='lower right', framealpha=0.2, fontsize=9)
ax.grid(axis='x', linewidth=0.5)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'podium_probability.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('Saved: output/podium_probability.png')

## 6. Individual Driver Deep Dive

Distribution of finishing positions for a single driver. Change `DRIVER` to any driver code in your dataset.

In [ ]:
DRIVER = 'VER'  # <-- change this to any driver code

if DRIVER not in standings_prob.index:
    print(f"Driver '{DRIVER}' not found. Available: {list(standings_prob.index)}")
else:
    driver_data = standings_prob.loc[DRIVER]
    numeric_data = driver_data[[c for c in driver_data.index if c.isdigit()]]
    dnf_prob = driver_data.get('DNF', 0)

    exp_pos = np.dot(numeric_data.values, [int(c) for c in numeric_data.index])

    fig, ax = plt.subplots(figsize=(13, 5))

    bar_colors = [F1_RED if p == numeric_data.idxmax() else '#444444' for p in numeric_data.index]
    bars = ax.bar(numeric_data.index, numeric_data.values * 100,
                  color=bar_colors, edgecolor='none', width=0.7)

    for bar, val in zip(bars, numeric_data.values):
        if val >= 0.01:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                    f'{val*100:.1f}%', ha='center', va='bottom', fontsize=7.5, color='white')

    ax.axvline(x=str(int(round(exp_pos))), color=F1_SILVER, linestyle='--',
               linewidth=1.2, label=f'Expected position: {exp_pos:.1f}')

    ax.set_title(f'{DRIVER} — Championship Position Probability Distribution\n'
                 f'DNF probability: {dnf_prob*100:.1f}%  |  Expected finish: P{exp_pos:.1f}',
                 fontsize=12, pad=12, color='white')
    ax.set_xlabel('Championship Position', fontsize=10)
    ax.set_ylabel('Probability (%)', fontsize=10)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.legend(fontsize=9, framealpha=0.2)
    ax.grid(axis='y', linewidth=0.5)
    ax.set_axisbelow(True)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'{DRIVER}_distribution.png', dpi=150,
                bbox_inches='tight', facecolor='#0f0f0f')
    plt.show()
    print(f'Saved: output/{DRIVER}_distribution.png')

## 7. Points Standings Comparison — Forecast vs Current

Compares the two simulation outputs side by side. Useful when one CSV is a mid-season snapshot and the other is end-of-season forecast.

In [ ]:
# Expected position for each driver from both outputs
def expected_position(df):
    numeric_cols = [c for c in df.columns if c.isdigit()]
    positions = np.array([int(c) for c in numeric_cols])
    return df[numeric_cols].apply(lambda row: np.dot(row.values, positions), axis=1)

exp_standings = expected_position(standings_prob).sort_values()
exp_simulated = expected_position(simulated_standings).reindex(exp_standings.index)

fig, ax = plt.subplots(figsize=(11, 8))

x = np.arange(len(exp_standings))
width = 0.38

ax.barh(x + width/2, exp_standings.values, width, color=F1_RED,    label='standings_prob',      edgecolor='none')
ax.barh(x - width/2, exp_simulated.values, width, color=F1_SILVER, label='simulated_standings',  edgecolor='none', alpha=0.75)

ax.set_yticks(x)
ax.set_yticklabels(exp_standings.index, fontsize=9)
ax.invert_xaxis()
ax.set_xlabel('Expected Championship Position (lower = better)', fontsize=10)
ax.set_title('Expected Championship Position — Simulation Comparison',
             fontsize=12, pad=14, color='white')
ax.legend(fontsize=9, framealpha=0.2)
ax.grid(axis='x', linewidth=0.5)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'simulation_comparison.png', dpi=150,
            bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('Saved: output/simulation_comparison.png')

---

## Summary

| Chart | File | Description |
|---|---|---|
| Win Probability | `win_probability.png` | P1 championship probability per driver |
| Full Heatmap | `full_heatmap.png` | All drivers × all positions probability matrix |
| Podium Probability | `podium_probability.png` | P1+P2+P3 stacked by contribution |
| Driver Deep Dive | `{DRIVER}_distribution.png` | Single driver position distribution |
| Simulation Comparison | `simulation_comparison.png` | Both CSVs side by side |

All charts are saved to `output/` and can be used directly in LinkedIn posts or reports.